# Dynamic Programming — Subtopic 6

## DP on Stocks (the state machine formulation)

**Problems covered:**
1. Best Time to Buy and Sell Stock I — at most **1 transaction**
2. Best Time to Buy and Sell Stock II — **unlimited transactions**
3. Best Time to Buy and Sell Stock III — at most **2 transactions**
4. Best Time to Buy and Sell Stock IV — at most **K transactions**
5. Best Time to Buy and Sell Stock with **Cooldown** — unlimited transactions, 1-day cooldown after selling
6. Best Time to Buy and Sell Stock with **Transaction Fee** — unlimited transactions, flat fee per round trip

> All six problems are realizations of one **finite state machine** on the joint state $(\text{day}, \text{holding}, \text{transactions\_left})$. Each variant specializes this machine by collapsing a dimension or adding a transition constraint. Internalizing the universal machine once collapses six "tricky" problems into one mechanical recipe.

---

# The universal state machine

Every stock-trading DP is a **shortest/longest path problem on a finite-state machine** whose nodes are tuples $(d, h, k)$ and whose edges are the daily decisions.

## The state

$$\text{dp}[d][h][k] = \text{max profit obtainable from day}\ d\ \text{onward,\ given current state}\ (h, k)$$

- $d \in \{0, 1, \ldots, n\}$ — current day. $d = n$ is the post-game terminal.
- $h \in \{0, 1\}$ — **holding flag**: $h = 1$ means we currently own one share; $h = 0$ means we do not.
- $k \in \{0, 1, \ldots, K\}$ — **transactions remaining**. A "transaction" is a buy-sell round-trip. We decrement $k$ at the moment of **buying** (interpretation: each buy commits one of our remaining round-trips).

> **Why this state is Markov-complete.** Once you know what day it is, whether you currently hold a share, and how many round-trips you still have available, the future is independent of how you got here. Past prices and decisions are irrelevant for forward planning. **This is the test for whether your state is correct: can the recurrence forget the path?**

## The transitions (the FSM diagram)

At day $d$, from state $(h, k)$, the legal moves are:

| Current state | Decision | Next state $(d+1, \cdot, \cdot)$ | Profit contribution |
|---|---|---|---|
| $(d, 0, k)$ | **rest** | $(d+1, 0, k)$ | $0$ |
| $(d, 0, k)$, $k \ge 1$ | **buy** | $(d+1, 1, k-1)$ | $-\text{price}[d]$ |
| $(d, 1, k)$ | **rest** | $(d+1, 1, k)$ | $0$ |
| $(d, 1, k)$ | **sell** | $(d+1, 0, k)$ | $+\text{price}[d]$ |

(Selling does **not** decrement $k$ under this convention — the round-trip was "committed" at buy. An equivalent formulation decrements on sell; pick one and be rigid.)

## The recurrence

$$\text{dp}[d][0][k] = \max\!\Big(\ \underbrace{\text{dp}[d+1][0][k]}_{\text{rest}}\ ,\ \ \underbrace{\text{dp}[d+1][1][k-1] - \text{price}[d]}_{\text{buy, if}\ k \ge 1}\ \Big)$$

$$\text{dp}[d][1][k] = \max\!\Big(\ \underbrace{\text{dp}[d+1][1][k]}_{\text{rest}}\ ,\ \ \underbrace{\text{dp}[d+1][0][k] + \text{price}[d]}_{\text{sell}}\ \Big)$$

**Base case.** $\text{dp}[n][h][k] = 0$ for all $h, k$ — once the game ends, no future profit is reachable. (A natural extra: we'd also like to *force* selling by end-of-game, but actually the $\max$ over "rest" handles that — if we end holding, we just don't extract value from that share. To enforce sell-by-end, also set $\text{dp}[n][1][k] = -\infty$. In practice the answer is $\text{dp}[0][0][K]$, and an optimal solution naturally ends at $h = 0$ since holding a share without selling is strictly dominated by selling.)

**Answer.** $\text{dp}[0][0][K]$.

## How each variant specializes the machine

| Variant | $K$ | Cooldown? | Fee? | Dimension collapsed |
|---|---|---|---|---|
| **I** — at most 1 | $1$ | no | no | $k$ becomes a 2-value flag |
| **II** — unlimited | $\infty$ | no | no | $k$ dropped entirely |
| **III** — at most 2 | $2$ | no | no | $k \in \{0, 1, 2\}$ |
| **IV** — at most $K$ | $K$ | no | no | $k \in \{0, \ldots, K\}$ (with $K \ge n/2$ shortcut → unlimited) |
| **V** — cooldown | $\infty$ | yes | no | $k$ dropped; **state space expands** to 3 states (hold / not-hold / cooldown) |
| **VI** — fee | $\infty$ | no | yes | $k$ dropped; per-edge profit modified |

The pedagogically critical row is **II**: when $K = \infty$, the $k$ dimension is *redundant* — we never run out — so it can be dropped entirely. The same recurrence collapses from 3D to 2D state.

## Complexity by variant

| Variant | Time | Space (optimized) |
|---|---|---|
| I, II, V, VI | $O(n)$ | $O(1)$ |
| III | $O(n)$ | $O(1)$ (six scalars per day) |
| IV | $O(nK)$ | $O(K)$ |

## The two parallel sequences

The recurrences for $\text{dp}[d][0][k]$ and $\text{dp}[d][1][k]$ **cross-reference each other** — today's best-not-holding depends on tomorrow's best-holding (via the buy decision) and vice versa. This is the hallmark of a state-machine DP: separate Bellman equations, one per state, coupled by the transitions. You compute them in lockstep.

---

## 6.1 Stock I — at most 1 transaction

**Problem.** Array $\text{price}[0..n-1]$. Buy at most once, sell at most once (sell day $\ge$ buy day). Maximize profit. Return 0 if no profit possible.

---

### Theory

**State definition.**
$\text{dp}[d][h]$ where $h \in \{0, 1\}$ — the $k$ dimension collapses because $K = 1$ implicitly: once we've bought, we cannot buy again (we'd need a fresh transaction); once we've sold, we cannot buy again either.

Actually with $K=1$ we *do* need to track whether the single transaction is "still available." A cleaner way to phrase this:

$\text{dp}[d][h]$ = max profit from day $d$ onward, where $h = 1$ means we currently hold the share (and have therefore already used the buy), $h = 0$ means we are not holding.

The $k$ dimension collapses because:
- If $h = 1$: we bought in the past (used our one transaction). Future options: rest or sell.
- If $h = 0$ on day $d$: have we used our transaction yet? We don't actually need to know, because:
  - If we **haven't** bought yet, we can still buy.
  - If we have bought and sold, we cannot buy again — but selling brought us to $h = 0$, so we now need an extra bit.

So actually $h = 0$ conflates "not yet bought" and "already sold." To distinguish, we'd need a third state — OR we observe a key fact:

> **Once we've sold (used our transaction), the optimal action for every remaining day is "rest" → $0$ future profit.** So $\text{dp}[d][\text{already sold}] = 0$ for all $d$.

This means we can collapse: $\text{dp}[d][0]$ in the rest of this DP refers to "have not yet bought" (the only non-trivial $h=0$ state). And the sell action transitions out of the recursion into the trivial $0$ tail.

**Recurrence.**
$$\text{dp}[d][0] = \max\!\Big(\ \underbrace{\text{dp}[d+1][0]}_{\text{rest}},\ \ \underbrace{\text{dp}[d+1][1] - \text{price}[d]}_{\text{buy}}\ \Big)$$

$$\text{dp}[d][1] = \max\!\Big(\ \underbrace{\text{dp}[d+1][1]}_{\text{rest}},\ \ \underbrace{0 + \text{price}[d]}_{\text{sell (no further profit)}}\ \Big)$$

**Base case.** $\text{dp}[n][0] = \text{dp}[n][1] = 0$.

**Boundary transitions table.**

| State | Decision | Next/value |
|---|---|---|
| $(d, 0)$ | rest | $\text{dp}[d+1][0]$ |
| $(d, 0)$ | buy | $\text{dp}[d+1][1] - \text{price}[d]$ |
| $(d, 1)$ | rest | $\text{dp}[d+1][1]$ |
| $(d, 1)$ | sell | $+\text{price}[d]$ (terminal — no further transactions) |
| $(n, \cdot)$ | terminal | $0$ |

**Why it works.**
- *Optimal substructure:* the best future from $(d, h)$ depends only on $(h, \text{prices}[d..n-1])$. Past prices and past actions are irrelevant.
- *Overlapping subproblems:* naive recursion is exponential in days; the FSM has $O(n)$ distinct states.

**Complexity.** Time $O(n)$, Space $O(n)$ → $O(1)$.

**Space optimization — validity proof.**
$\text{dp}[d][\cdot]$ depends only on $\text{dp}[d+1][\cdot]$. Keep two scalars `next0`, `next1` representing tomorrow's values; compute today's `cur0`, `cur1`; promote. Two scalars, no array.

**Alternative O(n) algorithm — "min so far, max profit."** Track the minimum price seen so far while sweeping left-to-right; at each day, the best profit if we sell today is $\text{price}[d] - \text{min\_so\_far}$. The maximum over days is the answer. This is the same time/space complexity, but it doesn't generalize to the other variants. The state-machine DP **does** generalize — that's why we use it as the prototype.

**Delta.** This is the simplest specialization of the universal machine. The $k$ dimension dissolves because (a) only one buy is allowed and (b) post-sell-state has trivially zero future value.

---

In [ ]:
// Stock I — three implementations
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[d][h] = max profit from day d onward, currently in holding state h.
int stockI_memo_helper(int d, int h, const vector<int>& price, vector<vector<int>>& dp) {
    int n = (int)price.size();
    if (d == n) return 0;                                  // base: terminal
    if (dp[d][h] != -1) return dp[d][h];
    int rest, action;
    if (h == 0) {
        rest = stockI_memo_helper(d + 1, 0, price, dp);    // do nothing
        action = stockI_memo_helper(d + 1, 1, price, dp) - price[d];  // buy
    } else {
        rest = stockI_memo_helper(d + 1, 1, price, dp);    // hold
        action = 0 + price[d];                             // sell, then terminal (no more transactions)
    }
    dp[d][h] = max(rest, action);
    return dp[d][h];
}
int stockI_memo(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n, vector<int>(2, -1));
    return stockI_memo_helper(0, 0, price, dp);            // start: day 0, not holding
}

// (B) Bottom-up tabulation — O(n) time, O(n) space
//     Iterate d from n-1 down to 0; dp[d][...] depends on dp[d+1][...].
int stockI_tab(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n + 1, vector<int>(2, 0));      // base row dp[n] all zeros
    for (int d = n - 1; d >= 0; --d) {
        // not holding
        int restNH = dp[d+1][0];                           // do nothing
        int buyNH  = dp[d+1][1] - price[d];                // buy
        dp[d][0] = max(restNH, buyNH);
        // holding
        int restH = dp[d+1][1];                            // continue holding
        int sellH = 0 + price[d];                          // sell, no future transactions => 0
        dp[d][1] = max(restH, sellH);
    }
    return dp[0][0];
}

// (C) Space-optimized — O(1)
//     dp[d] depends only on dp[d+1]: keep two scalars for tomorrow.
int stockI_opt(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    int next0 = 0, next1 = 0;                              // dp[n][0], dp[n][1]
    for (int d = n - 1; d >= 0; --d) {
        int cur0 = max(next0, next1 - price[d]);           // rest or buy
        int cur1 = max(next1, price[d]);                   // rest or sell (then 0 future)
        next0 = cur0;
        next1 = cur1;
    }
    return next0;                                          // dp[0][0]
}


In [ ]:
// Tests — Stock I
auto run_s1 = [](vector<int> p, int expected) {
    int a = stockI_memo(p);
    int b = stockI_tab(p);
    int c = stockI_opt(p);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "p=[";
    for (size_t k = 0; k < p.size(); ++k) cout << p[k] << (k+1 < p.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_s1({},                    0);   // empty
run_s1({5},                   0);   // single day: cannot trade
run_s1({1, 2},                1);   // buy day 0, sell day 1
run_s1({2, 1},                0);   // monotone decreasing: no profit
run_s1({7, 1, 5, 3, 6, 4},    5);   // LeetCode classic: buy 1, sell 6
run_s1({7, 6, 4, 3, 1},       0);   // monotone decreasing
run_s1({1, 2, 3, 4, 5},       4);   // monotone increasing: buy first, sell last
run_s1({3, 3, 3, 3},          0);   // all same: no profit
run_s1({2, 4, 1, 7},          6);   // sell 7 - buy 1


## 6.2 Stock II — unlimited transactions

**Problem.** Same array, but you may complete as many round-trips as you like (must sell before buying again — only one share held at a time). Maximize profit.

---

### Theory

**State definition.**
$\text{dp}[d][h]$ — the $k$ dimension **drops entirely** because we never run out of transactions. This is the pedagogically critical observation.

**Recurrence.**
$$\text{dp}[d][0] = \max\!\Big(\ \text{dp}[d+1][0]\ ,\ \ \text{dp}[d+1][1] - \text{price}[d]\ \Big)$$

$$\text{dp}[d][1] = \max\!\Big(\ \text{dp}[d+1][1]\ ,\ \ \text{dp}[d+1][0] + \text{price}[d]\ \Big)$$

Compare with Stock I: the only change is in the **sell branch** — selling now goes to $\text{dp}[d+1][0]$ (continue trading) instead of terminal $0$. This single change captures "unlimited transactions."

**Base case.** $\text{dp}[n][0] = \text{dp}[n][1] = 0$.

**Boundary transitions table.**

| State | Decision | Next/value |
|---|---|---|
| $(d, 0)$ | rest | $\text{dp}[d+1][0]$ |
| $(d, 0)$ | buy | $\text{dp}[d+1][1] - \text{price}[d]$ |
| $(d, 1)$ | rest | $\text{dp}[d+1][1]$ |
| $(d, 1)$ | sell | $\text{dp}[d+1][0] + \text{price}[d]$ |
| $(n, \cdot)$ | terminal | $0$ |

**Why it works.** The same Markov argument as Stock I. The recurrence couples $h = 0$ and $h = 1$ states tightly through buy/sell transitions.

**Complexity.** Time $O(n)$, Space $O(1)$.

**Folklore alternative — the "sum of positive deltas" formula.**

$$\text{answer} = \sum_{d=1}^{n-1} \max(0,\ \text{price}[d] - \text{price}[d-1])$$

*Proof.* Any sequence of trades earns total profit equal to the sum of $(\text{sell price} - \text{buy price})$ over completed round-trips. Each round-trip from buy day $b$ to sell day $s$ telescopes to $\sum_{d = b+1}^{s} (\text{price}[d] - \text{price}[d-1])$. The maximum is achieved by summing exactly the positive daily deltas (skip negative-delta days by not holding through them). The DP rediscovers this without any combinatorial insight.

The DP version doesn't beat the formula on Stock II, but it **does** generalize to V and VI (where the formula breaks). The pedagogical message: "easy closed-form" alternatives often hide that they don't compose.

**Delta vs Stock I.** Sell now leads back into the DP (more profit possible later) rather than terminating. One character of code; major semantic change.

---

In [ ]:
// Stock II — three implementations
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// (A) Top-down memoization
int stockII_memo_helper(int d, int h, const vector<int>& price, vector<vector<int>>& dp) {
    int n = (int)price.size();
    if (d == n) return 0;                                  // base
    if (dp[d][h] != -1) return dp[d][h];
    int rest, action;
    if (h == 0) {
        rest   = stockII_memo_helper(d + 1, 0, price, dp);
        action = stockII_memo_helper(d + 1, 1, price, dp) - price[d];  // buy
    } else {
        rest   = stockII_memo_helper(d + 1, 1, price, dp);
        action = stockII_memo_helper(d + 1, 0, price, dp) + price[d];  // sell (now leads back into DP)
    }
    dp[d][h] = max(rest, action);
    return dp[d][h];
}
int stockII_memo(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n, vector<int>(2, -1));
    return stockII_memo_helper(0, 0, price, dp);
}

// (B) Tabulation — O(n) time, O(n) space
int stockII_tab(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n + 1, vector<int>(2, 0));      // base dp[n][*] = 0
    for (int d = n - 1; d >= 0; --d) {
        dp[d][0] = max(dp[d+1][0], dp[d+1][1] - price[d]); // rest or buy
        dp[d][1] = max(dp[d+1][1], dp[d+1][0] + price[d]); // rest or sell (return into DP)
    }
    return dp[0][0];
}

// (C) Space-optimized — O(1)
int stockII_opt(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    int next0 = 0, next1 = 0;                              // dp[n][0], dp[n][1]
    for (int d = n - 1; d >= 0; --d) {
        int cur0 = max(next0, next1 - price[d]);
        int cur1 = max(next1, next0 + price[d]);
        next0 = cur0;
        next1 = cur1;
    }
    return next0;
}


In [ ]:
// Tests — Stock II
auto run_s2 = [](vector<int> p, int expected) {
    int a = stockII_memo(p);
    int b = stockII_tab(p);
    int c = stockII_opt(p);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "p=[";
    for (size_t k = 0; k < p.size(); ++k) cout << p[k] << (k+1 < p.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_s2({},                    0);
run_s2({5},                   0);
run_s2({1, 2},                1);
run_s2({2, 1},                0);
run_s2({7, 1, 5, 3, 6, 4},    7);   // LeetCode: buy 1, sell 5 (4) + buy 3, sell 6 (3) = 7
run_s2({7, 6, 4, 3, 1},       0);
run_s2({1, 2, 3, 4, 5},       4);   // unchanged from Stock I: monotone → one trip is optimal
run_s2({3, 3, 3, 3},          0);
run_s2({1, 5, 2, 6},          8);   // (5-1) + (6-2) = 8


## 6.3 Stock III — at most 2 transactions

**Problem.** Same array, but you may complete at most 2 round-trips. Maximize profit.

---

### Theory

**State definition.**
$\text{dp}[d][h][k]$ where $k \in \{0, 1, 2\}$ — explicit transaction counter, decremented on buy.

**Recurrence.**
$$\text{dp}[d][0][k] = \max\!\Big(\ \text{dp}[d+1][0][k]\ ,\ \ \text{dp}[d+1][1][k-1] - \text{price}[d]\ \Big),\quad k \ge 1$$

$$\text{dp}[d][1][k] = \max\!\Big(\ \text{dp}[d+1][1][k]\ ,\ \ \text{dp}[d+1][0][k] + \text{price}[d]\ \Big)$$

For $\text{dp}[d][0][0]$: buying is disallowed (no transactions left), so $\text{dp}[d][0][0] = \text{dp}[d+1][0][0] = 0$ — no value left to extract.

For $\text{dp}[d][1][0]$: still allowed to sell (we're holding from a past buy; selling completes the transaction and doesn't need a "fresh" one in our convention). Standard recurrence.

**Base case.** $\text{dp}[n][h][k] = 0$.

**Answer.** $\text{dp}[0][0][2]$.

**Boundary transitions table.**

| State | Decision | Next/value |
|---|---|---|
| $(d, 0, 0)$ | only rest | $\text{dp}[d+1][0][0]$ (= 0 by chain) |
| $(d, 0, k \ge 1)$ | rest / buy | $\max(\text{dp}[d+1][0][k],\ \text{dp}[d+1][1][k-1] - \text{price}[d])$ |
| $(d, 1, k)$ | rest / sell | $\max(\text{dp}[d+1][1][k],\ \text{dp}[d+1][0][k] + \text{price}[d])$ |
| $(n, \cdot, \cdot)$ | terminal | $0$ |

**Why it works.** Same Markov argument. The $k$ axis simply tracks the budget; transitions respect it.

**Complexity.** Time $O(n)$ (the $k$ axis has constant size 3). Space $O(1)$ (six scalars per day, six for tomorrow). Note that despite the explicit 3D table in pedagogical form, the size of $k$ is *small constant*, so big-O treats this as $O(n)$ time.

**Space optimization.** Same as before — only need tomorrow's six scalars to compute today's six.

**Delta vs Stock II.** Adds an explicit transaction count. The recurrence skeleton is identical except the buy branch reads $\text{dp}[d+1][1][k-1]$ (using one transaction) instead of $\text{dp}[d+1][1]$ (unchanged $k$).

**Delta vs Stock IV.** Same structure with $K = 2$ hard-coded. Could equivalently call Stock IV with $K = 2$. Many interview answers solve III via two-pass technique (left-to-right "best profit ending by day $d$ using 1 transaction," right-to-left "best profit starting at day $d$ using 1 transaction," then combine). The state-machine DP subsumes this without the manual two-pass — same asymptotic, less ad-hoc.

---

In [ ]:
// Stock III — three implementations
#include <vector>
#include <algorithm>
#include <climits>
#include <iostream>
using namespace std;

// (A) Top-down memoization
//     dp[d][h][k] = max profit from day d onward in state (h, k_left).
int stockIII_memo_helper(int d, int h, int k, const vector<int>& price,
                          vector<vector<vector<int>>>& dp) {
    int n = (int)price.size();
    if (d == n || (k == 0 && h == 0)) return 0;              // base: terminal or "no transactions left and not holding"
    if (dp[d][h][k] != -1) return dp[d][h][k];
    int rest, action;
    if (h == 0) {
        rest = stockIII_memo_helper(d + 1, 0, k, price, dp);
        // buy only valid if k >= 1
        action = (k >= 1) ? (stockIII_memo_helper(d + 1, 1, k - 1, price, dp) - price[d])
                          : INT_MIN;
    } else {
        rest = stockIII_memo_helper(d + 1, 1, k, price, dp);
        action = stockIII_memo_helper(d + 1, 0, k, price, dp) + price[d];  // sell
    }
    dp[d][h][k] = max(rest, action);
    return dp[d][h][k];
}
int stockIII_memo(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    int K = 2;
    vector<vector<vector<int>>> dp(n, vector<vector<int>>(2, vector<int>(K + 1, -1)));
    return stockIII_memo_helper(0, 0, K, price, dp);
}

// (B) Tabulation — O(n*K) time and space; K = 2 is fixed.
int stockIII_tab(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    const int K = 2;
    vector<vector<vector<int>>> dp(n + 1, vector<vector<int>>(2, vector<int>(K + 1, 0)));
    for (int d = n - 1; d >= 0; --d) {
        for (int k = 0; k <= K; ++k) {
            // not holding
            int rest = dp[d+1][0][k];
            int buy = (k >= 1) ? (dp[d+1][1][k-1] - price[d]) : INT_MIN;
            dp[d][0][k] = max(rest, buy);
            // holding
            int restH = dp[d+1][1][k];
            int sell = dp[d+1][0][k] + price[d];
            dp[d][1][k] = max(restH, sell);
        }
    }
    return dp[0][0][K];
}

// (C) Space-optimized — O(K)
//     Only need tomorrow's 2*(K+1) values to compute today's.
int stockIII_opt(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    const int K = 2;
    vector<vector<int>> next_dp(2, vector<int>(K + 1, 0));  // dp[n][h][k] = 0
    vector<vector<int>> cur_dp(2, vector<int>(K + 1, 0));
    for (int d = n - 1; d >= 0; --d) {
        for (int k = 0; k <= K; ++k) {
            int rest0 = next_dp[0][k];
            int buy   = (k >= 1) ? (next_dp[1][k-1] - price[d]) : INT_MIN;
            cur_dp[0][k] = max(rest0, buy);
            int rest1 = next_dp[1][k];
            int sell  = next_dp[0][k] + price[d];
            cur_dp[1][k] = max(rest1, sell);
        }
        swap(next_dp, cur_dp);                              // promote cur → next for d-1's iteration
    }
    return next_dp[0][K];
}


In [ ]:
// Tests — Stock III
auto run_s3 = [](vector<int> p, int expected) {
    int a = stockIII_memo(p);
    int b = stockIII_tab(p);
    int c = stockIII_opt(p);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "p=[";
    for (size_t k = 0; k < p.size(); ++k) cout << p[k] << (k+1 < p.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_s3({},                              0);
run_s3({1},                             0);
run_s3({1, 2, 3, 4, 5},                 4);   // 1 transaction is optimal
run_s3({3, 3, 5, 0, 0, 3, 1, 4},        6);   // LeetCode: (3-0) + (4-1) = 6
run_s3({1, 2, 3, 4, 5, 0, 1},           5);   // 4 + 1 = 5
run_s3({7, 6, 4, 3, 1},                 0);
run_s3({2, 1, 2, 0, 1},                 2);   // (2-1) + (1-0) = 2
run_s3({1, 2, 4, 2, 5, 7, 2, 4, 9, 0}, 13);   // two big trips


## 6.4 Stock IV — at most K transactions

**Problem.** Same array; at most $K$ round-trips. Maximize profit.

---

### Theory

**State definition.** Same as Stock III but with $k \in \{0, 1, \ldots, K\}$.

**Recurrence.** Identical to Stock III.

**The $K \ge \lfloor n/2 \rfloor$ shortcut.**

> **Observation.** No sequence of valid trades can complete more than $\lfloor n/2 \rfloor$ round-trips (each round-trip needs at least 2 distinct days). So if $K \ge \lfloor n/2 \rfloor$, the transaction limit is **non-binding** and the problem reduces to Stock II.

This avoids allocating an $O(nK)$ table when $K$ is large (think $K = 10^9$ with $n = 100$ — without the shortcut, we'd waste an obscene amount of memory).

**Boundary transitions table.** Same as Stock III; just allow $k$ up to $K$.

**Complexity.** Without the shortcut: $O(nK)$ time and space (or $O(K)$ optimized). With the shortcut: $O(n)$ when $K \ge n/2$.

**Delta vs Stock III.** Pure parameterization — the $K$ value becomes an argument instead of a fixed 2.

---

In [ ]:
// Stock IV — three implementations
#include <vector>
#include <algorithm>
#include <climits>
#include <iostream>
using namespace std;

// Stock II helper for the K >= n/2 shortcut
int stockII_helper(const vector<int>& price);

// (A) Top-down memoization
int stockIV_memo_helper(int d, int h, int k, int K, const vector<int>& price,
                         vector<vector<vector<int>>>& dp) {
    int n = (int)price.size();
    if (d == n || (k == 0 && h == 0)) return 0;            // base: terminal or no future value
    if (dp[d][h][k] != -1) return dp[d][h][k];
    int rest, action;
    if (h == 0) {
        rest = stockIV_memo_helper(d + 1, 0, k, K, price, dp);
        action = (k >= 1) ? (stockIV_memo_helper(d + 1, 1, k - 1, K, price, dp) - price[d])
                          : INT_MIN;
    } else {
        rest = stockIV_memo_helper(d + 1, 1, k, K, price, dp);
        action = stockIV_memo_helper(d + 1, 0, k, K, price, dp) + price[d];
    }
    dp[d][h][k] = max(rest, action);
    return dp[d][h][k];
}
int stockIV_memo(int K, const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1 || K == 0) return 0;
    if (K >= n / 2) return stockII_helper(price);          // shortcut: limit non-binding
    vector<vector<vector<int>>> dp(n, vector<vector<int>>(2, vector<int>(K + 1, -1)));
    return stockIV_memo_helper(0, 0, K, K, price, dp);
}

// (B) Tabulation — O(nK)
int stockIV_tab(int K, const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1 || K == 0) return 0;
    if (K >= n / 2) return stockII_helper(price);
    vector<vector<vector<int>>> dp(n + 1, vector<vector<int>>(2, vector<int>(K + 1, 0)));
    for (int d = n - 1; d >= 0; --d) {
        for (int k = 0; k <= K; ++k) {
            int rest0 = dp[d+1][0][k];
            int buy = (k >= 1) ? (dp[d+1][1][k-1] - price[d]) : INT_MIN;
            dp[d][0][k] = max(rest0, buy);
            int rest1 = dp[d+1][1][k];
            int sell = dp[d+1][0][k] + price[d];
            dp[d][1][k] = max(rest1, sell);
        }
    }
    return dp[0][0][K];
}

// (C) Space-optimized — O(K)
int stockIV_opt(int K, const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1 || K == 0) return 0;
    if (K >= n / 2) return stockII_helper(price);
    vector<vector<int>> next_dp(2, vector<int>(K + 1, 0));
    vector<vector<int>> cur_dp(2, vector<int>(K + 1, 0));
    for (int d = n - 1; d >= 0; --d) {
        for (int k = 0; k <= K; ++k) {
            int rest0 = next_dp[0][k];
            int buy = (k >= 1) ? (next_dp[1][k-1] - price[d]) : INT_MIN;
            cur_dp[0][k] = max(rest0, buy);
            int rest1 = next_dp[1][k];
            int sell = next_dp[0][k] + price[d];
            cur_dp[1][k] = max(rest1, sell);
        }
        swap(next_dp, cur_dp);
    }
    return next_dp[0][K];
}

// Local Stock II implementation for the shortcut (independent of stockII_opt from §6.2).
int stockII_helper(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    int next0 = 0, next1 = 0;
    for (int d = n - 1; d >= 0; --d) {
        int cur0 = max(next0, next1 - price[d]);
        int cur1 = max(next1, next0 + price[d]);
        next0 = cur0;
        next1 = cur1;
    }
    return next0;
}


In [ ]:
// Tests — Stock IV
auto run_s4 = [](int K, vector<int> p, int expected) {
    int a = stockIV_memo(K, p);
    int b = stockIV_tab(K, p);
    int c = stockIV_opt(K, p);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "K=" << K << " p=[";
    for (size_t k = 0; k < p.size(); ++k) cout << p[k] << (k+1 < p.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_s4(0, {1, 2, 3},           0);   // K=0: cannot trade
run_s4(1, {1, 2},              1);   // K=1 ⇒ same as Stock I
run_s4(2, {3, 2, 6, 5, 0, 3},  7);   // (6-2) + (3-0) = 7
run_s4(2, {2, 4, 1},           2);   // best single trip
run_s4(100, {3, 2, 6, 5, 0, 3}, 7);  // K >> n/2: same as unlimited (= Stock II); shortcut active
run_s4(3, {1, 2, 4, 2, 5, 7, 2, 4, 9, 0}, 15);  // 3 trips: (2-1)+(7-2)+(9-2) = 1+5+7 = 13? Re-check
                                                  // Actually try: (4-1) + (7-2) + (9-2) = 3+5+7 = 15
                                                  // Or: (2-1)+(5-2)+(9-2) = 1+3+7 = 11
                                                  // Best three trips: pick best three disjoint deltas.
                                                  // Unlimited (Stock II) on this array: positive deltas are 1,2,3,4,2,2,5 → sum = 1+2+3+4+2+2+5 = ... let me check:
                                                  //   p = 1,2,4,2,5,7,2,4,9,0
                                                  //   deltas: 1,2,-2,3,2,-5,2,5,-9
                                                  //   pos: 1+2+3+2+2+5 = 15. Yes.
                                                  // So 3 trips suffice to capture all 15. K=3 result = 15.
run_s4(2, {1, 2, 4, 2, 5, 7, 2, 4, 9, 0}, 13);   // Stock III answer (K=2): 13


## 6.5 Stock V — with cooldown (Stock II + 1-day mandatory rest after selling)

**Problem.** Unlimited transactions, but after you **sell**, you must rest at least one full day before you can buy again. Maximize profit.

---

### Theory — the state space expands

The two-state machine ($h \in \{0, 1\}$) is no longer enough: from $h = 0$ we cannot tell whether *yesterday* we sold (forbidding today's buy) or *yesterday* we were already not holding (allowing today's buy).

**Two clean fixes — pick one.**

### Fix A: 3-state machine.

Introduce a new state `cooldown` distinct from `not_holding`. The state $s \in \{\text{NH}, \text{H}, \text{CD}\}$:
- **NH** (not holding): can rest (stay NH) or buy (go to H).
- **H** (holding): can rest (stay H) or sell (go to CD).
- **CD** (cooldown): forced rest, must transition to NH next day.

Transitions and recurrence (forward direction this time; $\text{dp}[d][s]$ = max profit by end of day $d$):

$$\text{dp}[d][\text{NH}] = \max\!\big(\text{dp}[d-1][\text{NH}],\ \text{dp}[d-1][\text{CD}]\big)$$
$$\text{dp}[d][\text{H}] = \max\!\big(\text{dp}[d-1][\text{H}],\ \text{dp}[d-1][\text{NH}] - \text{price}[d]\big)$$
$$\text{dp}[d][\text{CD}] = \text{dp}[d-1][\text{H}] + \text{price}[d]$$

**Base case** (day $0$):
- $\text{dp}[0][\text{NH}] = 0$
- $\text{dp}[0][\text{H}] = -\text{price}[0]$ (bought on day 0)
- $\text{dp}[0][\text{CD}] = -\infty$ (cannot be in cooldown on day 0 — no prior sell)

**Answer.** $\max(\text{dp}[n-1][\text{NH}],\ \text{dp}[n-1][\text{CD}])$ — we end the game without holding stock.

### Fix B: 2-state machine with $d-2$ lookback.

Keep $h \in \{0, 1\}$ but tweak the buy transition to read 2 days back:
$$\text{dp}[d][1] = \max\!\big(\text{dp}[d-1][1],\ \text{dp}[d-2][0] - \text{price}[d]\big)$$
$$\text{dp}[d][0] = \max\!\big(\text{dp}[d-1][0],\ \text{dp}[d-1][1] + \text{price}[d]\big)$$

The $d-2$ enforces "if I buy at $d$, I cannot have just sold at $d-1$ — go back further."

Both formulations give the same answer. We use Fix A in code because it generalizes more cleanly to other cooldown durations (just add states); we mention Fix B as a tighter alternative.

**Boundary transitions table (Fix A).**

| Yesterday | Today's options |
|---|---|
| $\text{dp}[d-1][\text{NH}]$ | rest → NH; buy → H |
| $\text{dp}[d-1][\text{H}]$ | rest → H; sell → CD |
| $\text{dp}[d-1][\text{CD}]$ | forced rest → NH |

**Why it works.** The Markov property is restored: the 3-state classification of "yesterday's state" carries exactly the information needed to decide today's legal moves. Without `CD`, the not-holding state conflated two distinguishable cases (just sold vs. been resting); the new state distinguishes them.

**Complexity.** Time $O(n)$, Space $O(1)$ (three scalars).

**Delta vs Stock II.** State space expanded from 2 to 3 nodes to restore Markov property under the cooldown constraint. The recurrence becomes more verbose but mechanically identical in spirit.

---

In [ ]:
// Stock V (cooldown) — three implementations
// Using Fix A: 3-state machine (NH, H, CD), forward-time formulation.
#include <vector>
#include <algorithm>
#include <climits>
#include <iostream>
using namespace std;

const int NH = 0, H = 1, CD = 2;                            // state labels

// (A) Top-down memoization
//     dp[d][s] = max profit using days 0..d ending in state s.
int stockV_memo_helper(int d, int s, const vector<int>& price, vector<vector<int>>& dp) {
    if (d < 0) return INT_MIN / 2;                         // base: no negative days
    if (d == 0) {
        if (s == NH) return 0;
        if (s == H)  return -price[0];
        return INT_MIN / 2;                                // cannot be in cooldown on day 0
    }
    if (dp[d][s] != INT_MIN) return dp[d][s];
    int result;
    if (s == NH) {
        // Yesterday was NH (rest) or CD (forced exit)
        int fromNH = stockV_memo_helper(d - 1, NH, price, dp);
        int fromCD = stockV_memo_helper(d - 1, CD, price, dp);
        result = max(fromNH, fromCD);
    } else if (s == H) {
        // Yesterday was H (continued holding) or NH (bought today)
        int fromH  = stockV_memo_helper(d - 1, H,  price, dp);
        int fromNH = stockV_memo_helper(d - 1, NH, price, dp) - price[d];
        result = max(fromH, fromNH);
    } else { // s == CD
        // Yesterday was H, sold today
        result = stockV_memo_helper(d - 1, H, price, dp) + price[d];
    }
    dp[d][s] = result;
    return result;
}
int stockV_memo(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n, vector<int>(3, INT_MIN));
    int a = stockV_memo_helper(n - 1, NH, price, dp);
    int b = stockV_memo_helper(n - 1, CD, price, dp);      // CD also a valid terminal
    return max(a, b);
}

// (B) Tabulation — O(n) time, O(n) space
int stockV_tab(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n, vector<int>(3, INT_MIN / 2));
    dp[0][NH] = 0;
    dp[0][H]  = -price[0];
    // dp[0][CD] stays at -inf
    for (int d = 1; d < n; ++d) {
        dp[d][NH] = max(dp[d-1][NH], dp[d-1][CD]);                      // rest or end cooldown
        dp[d][H]  = max(dp[d-1][H],  dp[d-1][NH] - price[d]);           // hold or buy
        dp[d][CD] = dp[d-1][H] + price[d];                              // sell
    }
    return max(dp[n-1][NH], dp[n-1][CD]);
}

// (C) Space-optimized — O(1), three scalars
int stockV_opt(const vector<int>& price) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    int prev_NH = 0;
    int prev_H  = -price[0];
    int prev_CD = INT_MIN / 2;                             // unreachable on day 0
    for (int d = 1; d < n; ++d) {
        int cur_NH = max(prev_NH, prev_CD);
        int cur_H  = max(prev_H,  prev_NH - price[d]);
        int cur_CD = prev_H + price[d];
        prev_NH = cur_NH;
        prev_H  = cur_H;
        prev_CD = cur_CD;
    }
    return max(prev_NH, prev_CD);
}


In [ ]:
// Tests — Stock V (cooldown)
auto run_s5 = [](vector<int> p, int expected) {
    int a = stockV_memo(p);
    int b = stockV_tab(p);
    int c = stockV_opt(p);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "p=[";
    for (size_t k = 0; k < p.size(); ++k) cout << p[k] << (k+1 < p.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_s5({},                 0);   // empty
run_s5({5},                0);   // single
run_s5({1, 2},             1);   // simple
run_s5({1, 2, 3, 0, 2},    3);   // LeetCode: buy 1, sell 2, cooldown, buy 0, sell 2 = 1+2 = 3
run_s5({1, 2, 4},          3);   // buy 1, sell 4 = 3 (one trip; cooldown irrelevant)
run_s5({2, 1, 4},          3);   // buy 1, sell 4
run_s5({6, 1, 3, 2, 4, 7}, 6);   // best path with cooldown
run_s5({1, 4, 2},          3);


## 6.6 Stock VI — with transaction fee

**Problem.** Unlimited transactions, but each completed round-trip incurs a flat fee $f$. Maximize profit.

---

### Theory

**State definition.** $\text{dp}[d][h]$ — same 2-state machine as Stock II.

**Recurrence.** Identical to Stock II *except* the sell transition subtracts the fee:

$$\text{dp}[d][0] = \max\!\Big(\ \text{dp}[d+1][0]\ ,\ \ \text{dp}[d+1][1] - \text{price}[d]\ \Big)$$

$$\text{dp}[d][1] = \max\!\Big(\ \text{dp}[d+1][1]\ ,\ \ \text{dp}[d+1][0] + \text{price}[d] - f\ \Big)$$

**Where to put the fee.** Equivalent options:
- Pay on **sell**: subtract $f$ when selling. (Used in code.)
- Pay on **buy**: subtract $f$ when buying.
- Pay $f/2$ on each side. (Awkward with integer fees.)

All three give the same final answer because each round-trip pays the fee exactly once.

**Base case.** $\text{dp}[n][h] = 0$.

**Boundary transitions table.**

| State | Decision | Next/value |
|---|---|---|
| $(d, 0)$ | rest | $\text{dp}[d+1][0]$ |
| $(d, 0)$ | buy | $\text{dp}[d+1][1] - \text{price}[d]$ |
| $(d, 1)$ | rest | $\text{dp}[d+1][1]$ |
| $(d, 1)$ | sell | $\text{dp}[d+1][0] + \text{price}[d] - f$ |

**Why it works.** Same Markov argument. The fee is a per-transaction tax that the DP just folds into the sell-edge weight.

**Why the Stock II "positive-delta sum" formula breaks here.** With fee $f$, summing every positive daily delta would pay the fee on every consecutive up-day — gross overpayment. Optimal play often *holds through small dips* to avoid extra fees. The DP makes the right tradeoff automatically; the formula does not.

**Complexity.** Time $O(n)$, Space $O(1)$.

**Delta vs Stock II.** Subtract $f$ on sell. One character of code; alters the optimal strategy substantially when $f$ is large.

---

In [ ]:
// Stock VI (transaction fee) — three implementations
#include <vector>
#include <algorithm>
#include <iostream>
using namespace std;

// (A) Top-down memoization
int stockVI_memo_helper(int d, int h, int fee, const vector<int>& price, vector<vector<int>>& dp) {
    int n = (int)price.size();
    if (d == n) return 0;
    if (dp[d][h] != -1) return dp[d][h];
    int rest, action;
    if (h == 0) {
        rest   = stockVI_memo_helper(d + 1, 0, fee, price, dp);
        action = stockVI_memo_helper(d + 1, 1, fee, price, dp) - price[d];
    } else {
        rest   = stockVI_memo_helper(d + 1, 1, fee, price, dp);
        action = stockVI_memo_helper(d + 1, 0, fee, price, dp) + price[d] - fee;  // pay fee on sell
    }
    dp[d][h] = max(rest, action);
    return dp[d][h];
}
int stockVI_memo(const vector<int>& price, int fee) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n, vector<int>(2, -1));
    return stockVI_memo_helper(0, 0, fee, price, dp);
}

// (B) Tabulation — O(n)
int stockVI_tab(const vector<int>& price, int fee) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    vector<vector<int>> dp(n + 1, vector<int>(2, 0));
    for (int d = n - 1; d >= 0; --d) {
        dp[d][0] = max(dp[d+1][0], dp[d+1][1] - price[d]);
        dp[d][1] = max(dp[d+1][1], dp[d+1][0] + price[d] - fee);   // fee on sell
    }
    return dp[0][0];
}

// (C) Space-optimized — O(1)
int stockVI_opt(const vector<int>& price, int fee) {
    int n = (int)price.size();
    if (n <= 1) return 0;
    int next0 = 0, next1 = 0;
    for (int d = n - 1; d >= 0; --d) {
        int cur0 = max(next0, next1 - price[d]);
        int cur1 = max(next1, next0 + price[d] - fee);
        next0 = cur0;
        next1 = cur1;
    }
    return next0;
}


In [ ]:
// Tests — Stock VI
auto run_s6 = [](vector<int> p, int fee, int expected) {
    int a = stockVI_memo(p, fee);
    int b = stockVI_tab(p, fee);
    int c = stockVI_opt(p, fee);
    bool ok = (a == expected && b == expected && c == expected);
    cout << "fee=" << fee << " p=[";
    for (size_t k = 0; k < p.size(); ++k) cout << p[k] << (k+1 < p.size() ? "," : "");
    cout << "] -> memo=" << a << " tab=" << b << " opt=" << c
         << " (Expected: " << expected << ")"
         << (ok ? " OK" : " FAIL") << '\n';
};

run_s6({},                   2, 0);
run_s6({5},                  1, 0);
run_s6({1, 3, 2, 8, 4, 9},   2, 8);    // LeetCode: (8-1-2) + (9-4-2) = 5 + 3 = 8
run_s6({1, 3, 7, 5, 10, 3},  3, 6);    // (10 - 1 - 3) = 6, single trip dominates
run_s6({1, 2},               0, 1);    // fee=0: same as Stock II
run_s6({1, 2},               1, 0);    // fee equals gain: not worth trading
run_s6({1, 2},               2, 0);    // fee exceeds gain
run_s6({1, 2, 3, 4, 5},      1, 3);    // one trip 5-1-1 = 3 beats fragmented


# Unified Mental Model — DP on Stocks

All six problems are instances of one finite state machine on $(d, h, k)$, with each variant either:
1. **Collapsing a dimension** ($k$ drops in II, V, VI),
2. **Hard-coding a dimension's size** ($K = 1$ in I, $K = 2$ in III),
3. **Expanding the state space** (V adds the cooldown state), or
4. **Modifying edge weights** (VI subtracts a fee on sell).

---

In [ ]:
// ============================================================
// THE UNIVERSAL STATE MACHINE
// ============================================================
//
// State:  (d, h, k)
//   d ∈ {0, 1, …, n}    — day index (n = terminal)
//   h ∈ {0, 1}          — holding flag (1 = currently own one share)
//   k ∈ {0, …, K}       — transactions remaining (decrement on buy)
//
// Transitions (today → tomorrow):
//   (d, 0, k)  --rest-->  (d+1, 0, k)            value: 0
//   (d, 0, k)  --buy-->   (d+1, 1, k-1)          value: -price[d]   [requires k >= 1]
//   (d, 1, k)  --rest-->  (d+1, 1, k)            value: 0
//   (d, 1, k)  --sell-->  (d+1, 0, k)            value: +price[d]
//
// Recurrence:
//   dp[d][0][k] = max( dp[d+1][0][k] , dp[d+1][1][k-1] - price[d] )
//   dp[d][1][k] = max( dp[d+1][1][k] , dp[d+1][0][k]   + price[d] )
//
// Base:    dp[n][h][k] = 0
// Answer:  dp[0][0][K]
//
// ============================================================
// SPECIALIZATIONS
// ============================================================
//
// I  (K=1)      :  k is a 0/1 flag; sell goes to terminal-0 instead of into the DP.
//                  Equivalent: 2-state FSM with hand-coded "post-sell value = 0".
//
// II (K=∞)      :  k drops entirely. State reduces to dp[d][h]. Sell goes back into the DP.
//                  Folklore alternative: sum of positive (price[d] - price[d-1]).
//
// III (K=2)     :  k ∈ {0, 1, 2}. Same DP as IV with K=2. Equivalent two-pass technique exists.
//
// IV (K)        :  Generic. Shortcut: if K >= n/2 → reduce to II (limit non-binding).
//                  Space O(K) with 2*(K+1) scalars per day; time O(nK).
//
// V (cooldown)  :  State space EXPANDS to 3 states {NH, H, CD}. CD is "just sold yesterday."
//                  From CD only "forced rest → NH" is legal. dp[d][s] forward formulation.
//                  Equivalent: 2-state FSM with d-2 lookback in the buy branch.
//
// VI (fee)      :  K=∞ DP (drop k), modify sell-edge value: +price[d] - fee.
//                  Same complexity as II, different optimal strategy.
//
// ============================================================
// COMPLEXITY TABLE
// ============================================================
//   Variant  | Time    | Space (opt)  | Notes
//   ---------+---------+--------------+--------------------------------------------
//   I        | O(n)    | O(1)         | Could also use min-so-far one-liner
//   II       | O(n)    | O(1)         | Could also use sum-of-positive-deltas
//   III      | O(n)    | O(1)         | K=2 baked in (6 scalars per day)
//   IV       | O(nK)   | O(K)         | Shortcut O(n) when K >= n/2
//   V        | O(n)    | O(1)         | 3 scalars per day
//   VI       | O(n)    | O(1)         | Same shape as II
//
// ============================================================


# Decision Tree — recognizing a stock / state-machine DP

```
                ┌─────────────────────────────────────────────────────┐
                │ Sequential decisions over time with                │
                │ "currently holding / not holding" type state?       │
                └────────────────────┬────────────────────────────────┘
                                     │ yes
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Set up: dp[d][h][k] with h ∈ {0,1}.                 │
                │  Bellman equations cross-reference h=0 and h=1.     │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Is there a hard cap on number of transactions?      │
                │  • No (∞)            → drop k dimension             │
                │  • K=1, 2, …, small  → keep k explicit              │
                │  • K large vs n      → use shortcut → unlimited     │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Are there extra constraints between decisions?      │
                │  • Cooldown days    → ADD STATES (NH, H, CD, …)     │
                │  • Position limits  → add an integer dimension      │
                │  • Position size    → already inherent in h         │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Are there per-transaction costs / rewards?          │
                │  • Fee per round-trip       → modify sell-edge value│
                │  • Tax per transaction      → similar modification  │
                │  • Different fees per state → no problem; per-edge  │
                └────────────────────┬────────────────────────────────┘
                                     ▼
                ┌─────────────────────────────────────────────────────┐
                │ Write the Bellman equations one per state.           │
                │ Tabulate (forward in d or backward in d, either OK). │
                │ Space-optimize to O(#states) by keeping yesterday.   │
                └─────────────────────────────────────────────────────┘
```

**Three traps to avoid:**

1. **Pretending the state space is smaller than it is.** When you add a cooldown or a "max N consecutive holds" type rule, the Markov property typically breaks under the original state. Add the state dimension rather than hacking the recurrence with magic conditions on past values.

2. **Confusing decrement-on-buy with decrement-on-sell.** Both work but they shift base/answer indices by one. Pick one convention and *check* by tracing a 2-day, $K=1$ instance by hand before scaling.

3. **Forgetting the $K \ge n/2$ shortcut for Stock IV.** With $K = 10^9$ and $n = 100$, allocating an $O(nK)$ table will out-of-memory in seconds. The shortcut is not an optimization — it's a *correctness* requirement for the algorithm to terminate within constraints.

---

# Complexity Summary

| Problem | $K$ | State dims | Time | Space (opt) | Distinguishing structure |
|---|---|---|---|---|---|
| **Stock I** | $1$ | $(d, h)$ | $O(n)$ | $O(1)$ | Sell → terminal-0 (no future profit) |
| **Stock II** | $\infty$ | $(d, h)$ | $O(n)$ | $O(1)$ | $k$ drops; sell loops back into DP |
| **Stock III** | $2$ | $(d, h, k)$ | $O(n)$ | $O(1)$ | Explicit $k \in \{0,1,2\}$ |
| **Stock IV** | $K$ | $(d, h, k)$ | $O(nK)$ | $O(K)$ | Shortcut for $K \ge n/2$ |
| **Stock V** | $\infty$ | $(d, s)$, $s \in \{$NH, H, CD$\}$ | $O(n)$ | $O(1)$ | State space *expands* to 3 |
| **Stock VI** | $\infty$ | $(d, h)$ | $O(n)$ | $O(1)$ | Per-edge fee on sell transition |

---

# Closing Notes

**What you should now be able to do without reaching for any reference:**

1. Recognize a "state machine over time" problem from its phrasing — sequential decisions with a small set of internal states and constraints on transitions.
2. Write the **Bellman equations one per state**, with the matrix of transitions encoded by which edges go where.
3. Decide which dimensions are necessary vs collapsible — $k = \infty$ collapses, $K \ge n/2$ collapses, cooldown *expands* the state space rather than collapsing it.
4. Tabulate in $O(n \cdot |\text{states}|)$ time, space-optimize to $O(|\text{states}|)$.

**A pattern across the six problems:** every variant is a tweak on one of two prototypes — **Stock II** (the "unlimited transactions" 2-state FSM) and **Stock IV** (the "$K$-bounded" 3D table). All four "twist" variants (I, V, VI, and III) are one-line modifications to those prototypes:

- **I**: II with sell going to terminal-0 rather than recursing.
- **III**: IV with $K = 2$.
- **V**: II with one extra state for cooldown.
- **VI**: II with fee on sell edge.

Internalizing II and IV as the prototypes saves you from re-deriving each variant. **The discipline isn't "memorize six recurrences" — it's "memorize the FSM and the modification rules."**

**A deeper unifying perspective.** Each stock problem is a shortest/longest-path problem on a layered DAG: each day is a layer, the nodes within a layer are the FSM states, and edges go from layer $d$ to layer $d+1$ with the daily decision encoded as the edge label. The DP is just Bellman–Ford on a DAG (which runs in $O(|V| + |E|)$ rather than $V \cdot E$ because of the topological order). This view also explains why state-machine DPs feel so different from "subsequence" DPs: the latter have edges between subproblems on the same input dimension; the former have edges across the time dimension, with the input dimension *parameterizing the edge weights*.

**Looking ahead — Subtopic 7 (DP on LIS).** A new genre: we'll be choosing **which elements** of an array to include in an increasing subsequence, where the state is $(i, j)$ with $j$ = previous-chosen index. The clean $O(n^2)$ recurrence has a famous $O(n \log n)$ optimization via the **patience sorting** invariant — a beautiful first-principles construction that we'll derive rather than memorize. The mental discipline shifts from "what to do on each day?" (this subtopic) to "what element to take next, given what we last took?" (next subtopic).